<a href="https://colab.research.google.com/github/padhisneha2025-dev/python30/blob/main/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# file_organizer.py

import os
import shutil
import logging
import json
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict

# ── LOGGING ───────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("organizer.log"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ── TRIE FOR EXTENSION LOOKUP ─────────────────────────
class TrieNode:
    def __init__(self):
        self.children = {}
        self.category = None  # leaf node stores category name

class ExtensionTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, extension: str, category: str):
        """Insert extension → category mapping into trie"""
        node = self.root
        for char in extension.lower():
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.category = category

    def search(self, extension: str) -> str:
        """Search trie for extension, return category or 'Misc'"""
        node = self.root
        for char in extension.lower():
            if char not in node.children:
                return "Misc"
            node = node.children[char]
        return node.category or "Misc"

# ── FILE CATEGORIES ───────────────────────────────────
CATEGORIES = {
    "Images":     ["jpg", "jpeg", "png", "gif", "bmp", "svg", "webp", "ico"],
    "Videos":     ["mp4", "avi", "mov", "mkv", "wmv", "flv", "webm"],
    "Audio":      ["mp3", "wav", "flac", "aac", "ogg", "wma", "m4a"],
    "Documents":  ["pdf", "doc", "docx", "txt", "odt", "rtf", "tex"],
    "Spreadsheets": ["xls", "xlsx", "csv", "ods"],
    "Presentations": ["ppt", "pptx", "odp"],
    "Archives":   ["zip", "tar", "gz", "rar", "7z", "bz2"],
    "Code":       ["py", "js", "ts", "html", "css", "java", "cpp", "c",
                   "h", "rs", "go", "rb", "php", "swift", "kt", "sh"],
    "Data":       ["json", "xml", "yaml", "yml", "sql", "db", "sqlite"],
    "Executables": ["exe", "msi", "dmg", "apk", "deb", "rpm"],
    "Fonts":      ["ttf", "otf", "woff", "woff2"],
}

# ── STATS TRACKER ─────────────────────────────────────
@dataclass
class OrganizerStats:
    total_files: int = 0
    moved_files: int = 0
    skipped_files: int = 0
    errors: int = 0
    category_counts: dict = field(default_factory=lambda: defaultdict(int))
    start_time: str = field(default_factory=lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# ── CORE ORGANIZER ────────────────────────────────────
class FileOrganizer:
    def __init__(self, source_dir: str, dry_run: bool = False):
        self.source = Path(source_dir)
        self.dry_run = dry_run      # if True, simulate only
        self.trie = ExtensionTrie()
        self.stats = OrganizerStats()
        self.undo_log = []          # track moves for undo

        self._build_trie()

    def _build_trie(self):
        """Load all extensions into trie"""
        for category, extensions in CATEGORIES.items():
            for ext in extensions:
                self.trie.insert(ext, category)
        log.info(f"Trie built with {sum(len(v) for v in CATEGORIES.values())} extensions")

    def _get_unique_path(self, destination: Path) -> Path:
        """Handle filename conflicts — add (1), (2) etc."""
        if not destination.exists():
            return destination
        stem = destination.stem
        suffix = destination.suffix
        parent = destination.parent
        counter = 1
        while destination.exists():
            destination = parent / f"{stem} ({counter}){suffix}"
            counter += 1
        return destination

    def _move_file(self, file_path: Path, category: str):
        """Move a single file to its category folder"""
        dest_dir = self.source / category
        dest_file = self._get_unique_path(dest_dir / file_path.name)

        if not self.dry_run:
            dest_dir.mkdir(exist_ok=True)
            shutil.move(str(file_path), str(dest_file))
            self.undo_log.append((str(dest_file), str(file_path)))  # for undo

        self.stats.moved_files += 1
        self.stats.category_counts[category] += 1
        log.info(f"  {'[DRY RUN] ' if self.dry_run else ''}Moved: {file_path.name} → {category}/")

    def _organize_recursive(self, directory: Path, depth: int = 0):
        """Recursively organize files in directory and subdirectories"""
        indent = "  " * depth

        try:
            entries = list(directory.iterdir())
        except PermissionError:
            log.warning(f"{indent}Permission denied: {directory}")
            return

        for entry in entries:
            # skip already-created category folders at root level
            if entry.is_dir():
                if depth == 0 and entry.name in CATEGORIES:
                    continue
                log.info(f"{indent}📁 Entering: {entry.name}/")
                self._organize_recursive(entry, depth + 1)  # recursive call

            elif entry.is_file():
                self.stats.total_files += 1
                extension = entry.suffix.lstrip(".")

                if not extension:
                    category = "Misc"
                else:
                    category = self.trie.search(extension)  # O(len(ext)) lookup

                try:
                    self._move_file(entry, category)
                except Exception as e:
                    self.stats.errors += 1
                    log.error(f"  Failed to move {entry.name}: {e}")

    def organize(self):
        """Main entry point"""
        if not self.source.exists():
            log.error(f"Source directory does not exist: {self.source}")
            return

        mode = "DRY RUN" if self.dry_run else "LIVE"
        log.info(f"\n🗂️  File Organizer [{mode}]")
        log.info(f"   Source: {self.source}\n")

        self._organize_recursive(self.source)
        self._save_undo_log()
        self._print_stats()

    def undo(self):
        """Undo last organization using saved log"""
        undo_file = self.source / ".undo_log.json"
        if not undo_file.exists():
            log.warning("No undo log found")
            return

        with open(undo_file) as f:
            moves = json.load(f)

        reversed_moves = reversed(moves)
        count = 0
        for dest, original in reversed_moves:
            try:
                shutil.move(dest, original)
                count += 1
            except Exception as e:
                log.error(f"Undo failed for {dest}: {e}")

        log.info(f"↩️  Undid {count} file moves")
        undo_file.unlink()  # delete log after undo

    def _save_undo_log(self):
        """Save move history for undo"""
        if self.dry_run or not self.undo_log:
            return
        undo_file = self.source / ".undo_log.json"
        with open(undo_file, "w") as f:
            json.dump(self.undo_log, f, indent=2)
        log.info(f"  💾 Undo log saved → {undo_file}")

    def _print_stats(self):
        """Print organization summary"""
        print("\n" + "─" * 50)
        print("📊 ORGANIZATION SUMMARY")
        print("─" * 50)
        print(f"  Mode          : {'DRY RUN (no files moved)' if self.dry_run else 'LIVE'}")
        print(f"  Total files   : {self.stats.total_files}")
        print(f"  Moved         : {self.stats.moved_files}")
        print(f"  Errors        : {self.stats.errors}")
        print(f"\n  Files by category:")
        for cat, count in sorted(self.stats.category_counts.items()):
            bar = "█" * min(count, 30)
            print(f"    {cat:<15} {bar} {count}")
        print("─" * 50 + "\n")

# ── CLI INTERFACE ─────────────────────────────────────
def main():
    import sys

    print("\n🗂️  Bulk File Organizer")
    print("=" * 40)
    print("Commands:")
    print("  organize <path>          — organize a folder")
    print("  organize <path> --dry    — preview without moving")
    print("  undo <path>              — undo last organization")
    print("=" * 40 + "\n")

    if len(sys.argv) < 3:
        # interactive mode for Colab
        action = input("Action (organize/undo): ").strip().lower()
        path = input("Folder path: ").strip()
        dry = input("Dry run? (y/n): ").strip().lower() == "y"
    else:
        action = sys.argv[1]
        path = sys.argv[2]
        dry = "--dry" in sys.argv

    organizer = FileOrganizer(path, dry_run=dry)

    if action == "organize":
        organizer.organize()
    elif action == "undo":
        organizer.undo()
    else:
        print("Unknown command")

if __name__ == "__main__":
    main()


🗂️  Bulk File Organizer
Commands:
  organize <path>          — organize a folder
  organize <path> --dry    — preview without moving
  undo <path>              — undo last organization

Unknown command
